In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---
import IPython.display as ipd
from scipy.io import wavfile
from scipy import signal
from fractions import Fraction
import read_in_rsf as sf          # RSF reader in the repository root (for data/CAPshot)

def embed_audio(x, rate, max_rate=11025):
    """Audio player for a NumPy array, mono at <= 11.025 kHz (keeps the notebook small)."""
    x = np.asarray(x, dtype=float)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if rate > max_rate:
        r = Fraction(max_rate/rate).limit_denominator(1000)
        x = signal.resample_poly(x, r.numerator, r.denominator); rate = rate*r.numerator/r.denominator
    return ipd.Audio(x, rate=rate)

# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 5 &mdash; 1D Fourier Transforms</h1></center>

A Fourier series describes a signal that repeats forever, but most signals we record do not repeat: a piano note fades,
an earthquake arrives once, and a seismic shot record is a single transient. The Fourier transform extends the series to these
aperiodic signals by letting the period grow without bound, so the discrete set of harmonics becomes a continuous spectrum.
It is the language geophysicists use to talk about bandwidth, resolution and filtering. In this module we define the transform and
its inverse, compute the canonical pairs (boxcar and sinc, delta function, Gaussian, decaying exponential, Dirac comb), state
the properties that make it useful (above all the scaling theorem), and then apply it to musical instruments, an earthquake
and a land seismic shot record.

# Outline
- [5.1 Signals That Do Not Repeat: Instruments and an Earthquake](#5.1-Signals-That-Do-Not-Repeat:-Instruments-and-an-Earthquake)
- [5.2 The Fourier Transform](#5.2-The-Fourier-Transform)
  - [5.2.1 Power and magnitude spectra](#5.2.1-Power-and-magnitude-spectra)
  - [5.2.2 Alternate conventions](#5.2.2-Alternate-conventions)
- [5.3 A First Numerical Example: A Beat Pattern](#5.3-A-First-Numerical-Example:-A-Beat-Pattern)
  - [5.3.1 The frequency layout of np.fft.fft](#5.3.1-The-frequency-layout-of-np.fft.fft)
- [5.4 Which Functions Have a Fourier Transform? The Dirichlet Conditions](#5.4-Which-Functions-Have-a-Fourier-Transform?-The-Dirichlet-Conditions)
- [5.5 Fourier Transform Pairs I: The Boxcar and the Sinc](#5.5-Fourier-Transform-Pairs-I:-The-Boxcar-and-the-Sinc)
- [5.6 Properties of the Fourier Transform](#5.6-Properties-of-the-Fourier-Transform)
  - [5.6.1 Proofs of three properties](#5.6.1-Proofs-of-three-properties)
- [5.7 Fourier Transform Pairs II: The Delta Function, Cosine and Sine](#5.7-Fourier-Transform-Pairs-II:-The-Delta-Function,-Cosine-and-Sine)
- [5.8 Symmetry Properties](#5.8-Symmetry-Properties)
- [5.9 Fourier Transform Pairs III: Gaussian, One-Sided Exponential and Dirac Comb](#5.9-Fourier-Transform-Pairs-III:-Gaussian,-One-Sided-Exponential-and-Dirac-Comb)
- [5.10 From Continuous to Sampled: The DTFT and the DFT](#5.10-From-Continuous-to-Sampled:-The-DTFT-and-the-DFT)
- [5.11 A Geophysical Example: A Seismic Shot Record](#5.11-A-Geophysical-Example:-A-Seismic-Shot-Record)
- [5.12 Instruments and Earthquakes Revisited](#5.12-Instruments-and-Earthquakes-Revisited)
- [5.13 Parting Thoughts](#5.13-Parting-Thoughts)
- [Microquestions](#Microquestions)

<a name="5.1"></a>
## 5.1 Signals That Do Not Repeat: Instruments and an Earthquake

Let's begin our discussion of Fourier transforms by looking at a few interesting signals. Figure 5.1 shows four different musical instruments all playing the same note, C5 (523 Hz), one octave above middle C (C4, 262 Hz).

In [ ]:
## . . Load four instruments playing C5 (U. Iowa samples) and plot them
def plot_instrument(ax, xx, sr, title):
    tt = np.arange(0, np.size(xx, 0))/sr
    ax.plot(tt, xx, color=DSP_COLORS['time_domain'], lw=0.8)
    ax.set_title(title, fontsize=13); ax.set_ylabel('amplitude')

srp, piano   = wavfile.read('wav/piano-C5.wav')
srf, flute   = wavfile.read('wav/flute-C5.wav')
srt, trumpet = wavfile.read('wav/trumpet-C5.wav')
srv, violin  = wavfile.read('wav/violin-C5.wav')

## . . Subtract off mean value of each signal (i.e., remove DC component)
## . . Think about what '=' means in this context!
piano   = piano  -np.mean(piano  )
flute   = flute  -np.mean(flute  )
trumpet = trumpet-np.mean(trumpet)
violin  = violin -np.mean(violin )

## . . Not all of the files are of the same size ... so can only combine to the minimum file size
minsize = min([piano.size, flute.size, trumpet.size, violin.size])
## . . Think about what the command above is doing!

## . . The trumpet file has a lot of silence at the beginning: start 'tstart' samples in
tstart = 19000

## . . Superpose the different instruments (note that you are 'slicing' the arrays!)
orchestra = 2.*piano[1:minsize]+ \
               flute[1:minsize]+ \
               trumpet[tstart+1:minsize+tstart]+ \
               violin[1:minsize]

fig, axs = plt.subplots(5, 1, figsize=(12, 10), sharex=True)
plot_instrument(axs[0], 2*piano, srp, '(a) Piano - C5 (523 Hz)')
plot_instrument(axs[1], flute, srf, '(b) Flute - C5 (523 Hz)')
plot_instrument(axs[2], trumpet[tstart:], srt, '(c) Trumpet - C5 (523 Hz)')
plot_instrument(axs[3], violin, srv, '(d) Violin - C5 (523 Hz)')
plot_instrument(axs[4], orchestra, srp, '(e) Orchestra - C5 (523 Hz)')
axs[4].set_xlabel('time [s]'); axs[4].set_xlim(0, 7)
plt.tight_layout()
plt.show()

<b>Figure 5.1.</b> Four instruments playing C5 (523 Hz): (a) piano (doubled in amplitude), (b) flute, (c) trumpet, (d) violin, and (e) an "orchestra" that superposes (a)&ndash;(d). <i>(Python-generated from samples of the <a href="https://theremin.music.uiowa.edu/">U. Iowa Musical Instrument Samples</a>.)</i>

Let's now listen to them individually:

In [ ]:
## . . Listen to each instrument (players labeled as in Figure 5.1)
for label, xx, rate in [('(a) Piano', piano, srp), ('(b) Flute', flute, srf), ('(c) Trumpet (first 3 s)', trumpet[tstart:tstart + 3*srt], srt),
                        ('(d) Violin', violin, srv), ('(e) Orchestra', orchestra, srp)]:
    display(HTML(f'<b>{label}</b>'))
    display(embed_audio(xx, rate=rate))

Let's now look at a very different type of signal: an earthquake.  Here's an example of a vertical component of an earthquake recording:

In [ ]:
## . . Load a vertical component of an earthquake recording
sr, equake = wavfile.read('wav/equake4.wav')
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(np.arange(equake.size)/sr, equake, color=DSP_COLORS['time_domain'], lw=0.6)
ax.set_xlabel('time [s]'); ax.set_ylabel('amplitude'); ax.set_xlim(0, equake.size/sr)
plt.tight_layout(); plt.show()

<b>Figure 5.2.</b> Vertical component of a recorded distant earthquake (normalized). <i>(Python-generated; audio generated from open-source seismic data distributed by IRIS.)</i>

Let's now listen to the earthquake:

In [ ]:
embed_audio(equake, rate=sr)

You'll obviously have heard that the earthquake sounds quite different than the musical instruments.  Why is this? How can we describe this mathematically? Why can't we use Fourier Series? Because this is an **aperiodic** signal, we must use a different tool: the **Fourier Transform**.

<a name="5.2"></a>
## 5.2 The Fourier Transform

[Fourier Transforms](https://en.wikipedia.org/wiki/Fourier_transform) (or FTs) are used to decompose **continuous-time aperiodic** signals into its constituent frequencies. (The pair of time $t$ and angular frequency $\omega$ are termed [conjugate variables](https://en.wikipedia.org/wiki/Conjugate_variables).) For example, think about decomposing a recorded piece of music into its individual notes (aka frequencies!) that make up the composition.  The magnitude of the FT then corresponds to magnitude of that frequency present in the composition.  The phase of the FT describes the offset of that frequency from the basic sinusoid.  Note that unlike the Fourier Series, FTs are defined on a continuous range of frequencies $\omega$.

The Fourier Transform (commonly denoted $\mathcal{F}$) of an **aperiodic** and **continuous time** signal $x(t)$ is given by:

<div class="alert alert-info">
$$
\widehat{X}(\omega) = \int_{-\infty}^{\infty} x(t)\,\mathrm{e}^{-i\omega t} \mathrm{d}t = \mathcal{F}[x(t)] \tag{1}
$$
</div>

where $\widehat{X}(\omega)$ is also called the Fourier representation of $x(t)$ in the frequency domain.  In this course I will denote functions in the conjugate Fourier domain by the $\LaTeX$ *wide-hat* symbol - e.g., $\widehat{\gamma}$. In general, $\widehat{X}(\omega)$ is a complex-valued function.

<b>Inverse Fourier transform.</b> The [inverse Fourier Transform](https://en.wikipedia.org/wiki/Fourier_inversion_theorem) (commonly denoted $\mathcal{F}^{-1}$) is given by:

<div class="alert alert-info">
$$
x(t) = \frac{1}{2\pi} \int_{-\infty}^{\infty} \widehat{X}(\omega) \, \mathrm{e}^{i \omega t} \,\mathrm{d}\omega = \mathcal{F}^{-1}[\widehat{X}(\omega)] \tag{2}
$$
</div>

Note that $\mathcal{F}^{-1}$ is defined this way such that the following holds:

$$
\begin{eqnarray}
\mathcal{F}^{-1} \left( \mathcal{F} g(t) \right) & = & g(t) \\
\mathcal{F} \left( \mathcal{F}^{-1} \widehat{G}(\omega) \right) &=& \widehat{G}(\omega) \\
\end{eqnarray}  \tag{3}
$$

Also, note the following 

$$ \mathcal{F}^0 = \mathrm{Iden}, \quad \mathcal{F}^{1} = \mathcal{F}, \quad \mathcal{F}^{2} = \mathcal{P}, \quad \mathcal{F}^{3}=\mathcal{F}^{-1}, \quad \mathcal{F}^{4} = \mathrm{Iden} \tag{4}
$$

where $\mathcal{P}$ is the **Parity operator** that inverts time: $\mathcal{P[f(t)]}\rightarrow f(-t)$.  Over your science and engineering careers you will no doubt grow to appreciate the invertibility of a wide number of transformations!

<div class="alert alert-info">
<b>Convention:</b> These notes use <b>angular frequency</b> throughout the analysis: $\widehat{X}(\omega)=\int x(t)\,e^{-i\omega t}\,dt$
(equation 1), with the factor $1/2\pi$ on the inverse (equation 2), the same convention as Module 3. Analytic formulas are written in
$\omega$ (rad/s). Numerical plots label their axes in hertz, $f=\omega/2\pi$, because <code>np.fft.fftfreq(n, d=dt)</code> returns cycles per
unit time. The alternatives in Section 5.2.2 are given only so that you recognize them in other books.
</div>

<div class="alert alert-info">
<b>Synthesis &mdash; from Fourier series to Fourier transform.</b> Multiply Module 4's analysis formula by the period $T$:
$TC_k=\int_{-T/2}^{T/2}x(t)e^{-ik\omega_0t}dt$. As $T\to\infty$ the line spacing $\omega_0=2\pi/T$ shrinks to $d\omega$, $k\omega_0$ becomes a
continuous $\omega$, $TC_k$ becomes $\widehat X(\omega)$, and the synthesis sum $\sum C_ke^{ik\omega_0t}=\tfrac{1}{2\pi}\sum (TC_k)e^{ik\omega_0t}\,\omega_0$
becomes the integral (2). The $1/2\pi$ in the inverse is the $1/T$ of the series in disguise, and the basis functions are still the complex
exponentials of Module 3 (MATH 348 calls this passage from series to integral the Fourier integral).
</div>

<a name="5.2.1"></a>
### 5.2.1 Power and magnitude spectra

You might often hear the term [Fourier Power Spectrum](https://en.wikipedia.org/wiki/Spectral_density) or something meant to be equivalent (e.g., energy spectral density or power spectral density).  When you hear this people are generally talking about the **power spectrum** $P(\widehat{X}(\omega) )$, which is given by:

$$ P(\widehat{X}(\omega) ) = \left|\widehat{X}(\omega) \right|^2 = \widehat{X}(\omega)\overline{ \widehat{X}(\omega)} \tag{5} $$

We will also frequently discuss the **magnitude spectrum** $M(\widehat{X}(\omega))$, which is simply the square root of the power spectrum 

$$ M(\widehat{X}(\omega) ) = \sqrt{\left|\widehat{X}(\omega) \right|^2} = \sqrt{\widehat{X}(\omega)\overline{ \widehat{X}(\omega)}} \tag{6} $$

There are some who sometimes use amplitude spectrum; however, because amplitudes can be both positive and negative (and potentially complex numbers) whereas magnitudes are always positive real numbers, in this course the preference will be **magnitude** over amplitude (of course unless we are referring specifically to an amplitude).

<a name="5.2.2"></a>
### 5.2.2 Alternate conventions

Note that, like the Fourier Series, there are alternate definitions of the Fourier Transform than that specified above. In particular, note the following:

$$ \widehat{F}(f) = \int_{-\infty}^{\infty} f(t)\,e^{-i2\pi f t} \mathrm{d}t \tag{7}$$

and

$$ f(t) = \int_{-\infty}^{\infty} \widehat{F}(f)\,e^{i2\pi f t} \mathrm{d}f \tag{8}$$

It is important to note that this alternate definition leads to different frequency-domain scaling than that discussed above.  Alternatively, one often sees a "symmetric version" of the Fourier Transform specified by angular frequency:


$$\widehat{X}(\omega) = \frac{1}{\sqrt{2\pi} }\int_{-\infty}^{\infty} x(t)\,\mathrm{e}^{-i\omega t} \mathrm{d}t  \tag{9}$$

and 

$$ x(t) = \frac{1}{\sqrt{2\pi}} \int_{-\infty}^{\infty} \widehat{X}(\omega) \, \mathrm{e}^{i \omega t} \,\mathrm{d}\omega \tag{10} $$

Some people really like symmetry...

<a name="5.3"></a>
## 5.3 A First Numerical Example: A Beat Pattern

Let's look at an example where we have a function

$$f(t) = 2\cos (2\pi f_1 t) + 4\sin (2\pi f_2 t) \tag{11}$$

For the discussion below, take note that this function is a combination of even ($\cos$) and odd ($\sin$) terms.  

Let's first plot this for two nearby frequencies where $f_1=3.95$ Hz and $f_2=4.05$ Hz.

In [ ]:
## . . Let's define a few key parameters
Fs = 100              # sampling rate (Hz)
Ts = 1.0/Fs           # sampling interval (s)
tmin, tmax = 0., 200  # start/stop times (s)
f1, f2 = 3.95, 4.05   # the two frequencies (Hz)

## . . Time vector, angular frequencies, and the signal y(t) of equation (9)
t = np.arange(tmin, tmax, Ts)
w1 = 2.0*np.pi*f1
w2 = 2.0*np.pi*f2
y = 2.*np.cos(w1*t) + 4.*np.sin(w2*t)

fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(t, y, color=DSP_COLORS['time_domain'], lw=1.0)
ax.set_xlabel('time [s]'); ax.set_ylabel('amplitude')
ax.set_xlim(tmin, 0.2*tmax); ax.set_ylim(1.05*y.min(), 1.05*y.max())
plt.tight_layout(); plt.show()

<b>Figure 5.3.</b> A beat pattern from the superposition of two sinusoids that are close in frequency, $f_1=3.95$ Hz and $f_2=4.05$ Hz (equation 11); the envelope repeats every $1/(f_2-f_1)=10$ s. The first 40 s of the 200 s record are shown. <i>(Python-generated.)</i>

Let's now listen to the superposed signals. The 100 samples/s record is played 44.1 times faster than real time, so the 3.95 and 4.05 Hz tones are heard at 174 and 179 Hz and the slow beat at 4.4 Hz:

In [ ]:
## . . Played 44.1 times faster than real time (4410 samples/s instead of 100), so 3.95/4.05 Hz become 174/179 Hz
embed_audio(y, rate=4410)

<div class="alert alert-info">
<b>From np.fft.fft to the Fourier transform:</b> <code>np.fft.fft(x)</code> computes a discrete Fourier transform of the samples. Divided by the number of samples $N$, it gives
Fourier-series-like amplitudes for a periodic signal, which is how Figure 5.4 is normalized. Multiplied by the sample interval $\Delta t$, it
approximates the continuous transform $\widehat{X}(\omega)$ of a transient, which is how Figure 5.6 is computed.
</div>

In [ ]:
def stem_plot(ax, f, g, ylabel, title):
    keep = np.abs(g) > 1e-6                           # . . only the nonzero bins (the rest are zero)
    markerline, stemlines, baseline = ax.stem(f[keep], g[keep])   # . . returns 3 objects
    ax.axhline(0, color='k', lw=1.5)
    plt.setp(baseline, 'visible', False)
    plt.setp(stemlines, 'color', 'k', 'linewidth', 1)
    plt.setp(markerline, 'color', DSP_COLORS['freq_domain'])
    ax.set_ylabel(ylabel); ax.set_title(title, fontsize=13)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)

## . . length of the signal y defined above, frequency vector, and the FFT normalized by n
n = len(y)
freqs = np.fft.fftfreq(n, Ts)
Y = np.fft.fft(y)/n

fig, axs = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True)
stem_plot(axs[0], freqs, np.real(Y), r'$\Re(Y)$', '(a) Real part: even contributions')
stem_plot(axs[1], freqs, np.imag(Y), r'$\Im(Y)$', '(b) Imaginary part: odd contributions')
stem_plot(axs[2], freqs, np.real(Y*np.conj(Y)), r'$|Y|^2$', '(c) Power spectrum')
axs[2].set_xlabel('frequency [Hz]')
plt.tight_layout(); plt.show()

<b>Figure 5.4.</b> Fourier components of the beat signal $y(t)=2\cos 2\pi(3.95)t + 4\sin 2\pi(4.05)t$ of Figure 5.3: (a) the real part, $+1$ at $\pm 3.95$ Hz; (b) the imaginary part, $-2$ at $+4.05$ Hz and $+2$ at $-4.05$ Hz; (c) the power spectrum. The real and imaginary contributions are half the amplitudes of the time-domain terms, and the power is the square of half the amplitude, because Euler's formula splits each sinusoid between positive and negative frequencies. <i>(Python-generated.)</i>

<a name="5.3.1"></a>
### 5.3.1 The frequency layout of np.fft.fft

One of the interesting things to get your head around is the layout of the frequency axis.  Let's first plot the *freqs* function from the above example.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
for ax, f, ttl in [(axs[0], freqs, '(a) Order returned by fftfreq'),
                   (axs[1], np.fft.fftshift(freqs), '(b) After fftshift'),
                   (axs[2], np.fft.fftshift(np.fft.fftshift(freqs)), '(c) After two fftshifts')]:
    ax.plot(f, color=DSP_COLORS['freq_domain'])
    ax.set_xlabel('index'); ax.set_title(ttl, fontsize=12)
axs[0].set_ylabel('frequency [Hz]')
plt.tight_layout(); plt.show()

<b>Figure 5.5.</b> The layout of the frequency axis returned by <code>np.fft.fftfreq</code> for the beat signal: (a) as returned, (b) after <code>np.fft.fftshift</code>, and (c) after applying <code>np.fft.fftshift</code> twice, which restores the original order for even $N$. <i>(Python-generated.)</i>

In the first subplot we see that the frequency layout isn't as one might expect. It actually goes from $[0,f_N)$ and then from $[-f_N,0)$, where $f_N$ is the Nyquist frequency, half the sampling rate.  To get these plotted in the regular layout from $[-f_{max},f_{max}]$ we have to use the *np.fft.fftshift()* command.  This is illustrated in panel (b).  If we apply the *np.fft.fftshift()* command again, we recover the original frequency order (for even $N$; use *np.fft.ifftshift()* in general).

The reason why we need this is that the output of *np.fft.fft()* has the order $[0,f_N)$ and then $[-f_N,0)$.  Thus, to plot a Fourier transform in the expected order you must do something like:

* *plt.plot(np.fft.fftshift(freqs),np.fft.fftshift(My_FT_signal))*

Note that the *np.fft.fftshift()* command can take 1D, 2D ... ND data!

<div class="alert alert-success">
<h3><center>Distraction A: Gauss had the FFT first</center></h3>

The fast algorithm behind <code>np.fft.fft</code> was published by J. W. Cooley and J. W.
Tukey in 1965, and it made routine spectral analysis of seismic data possible. C. F. Gauss had used an equivalent method around 1805, in
unpublished work on interpolating the orbits of asteroids; it appeared only in his collected works, after his death.

<i>References:</i> J. W. Cooley and J. W. Tukey (1965), "An algorithm for the machine calculation of complex Fourier series,"
<i>Mathematics of Computation</i> 19(90), 297&ndash;301. M. T. Heideman, D. H. Johnson and C. S. Burrus (1984),
<a href="https://www.cis.rit.edu/class/simg716/Gauss_History_FFT.pdf">"Gauss and the history of the fast Fourier transform,"</a>
<i>IEEE ASSP Magazine</i> 1(4), 14&ndash;21.
</div>

<a name="5.4"></a>
## 5.4 Which Functions Have a Fourier Transform? The Dirichlet Conditions

In the above examples we have taken the Fourier transform without really thinking about whether we can apply this transformation. This observation raises the following question: **can all functions be Fourier transformed?**
The short answer is no. Functions $f(t)$ and $\widehat{F}(\omega)$ must obey a number of conditions that are termed the **Dirichlet Conditions**. These are that a function must be:


(1) Absolutely integrable; i.e., $\int_{-\infty}^{\infty} |f(t)| \, \mathrm{d}t$ is finite, which is satisfied by signals that decay as $|t|\rightarrow \infty$
   * What is the consequence of a function not being absolutely integrable?
    
    
(2) Single valued
   * What is the consequence of a function not being single valued?
    
    
(3) Be piecewise continuous within the specified interval
   * Step, boxcar and absolute value functions are permitted 


(4) Have upper and lower bounds

Note that the $\delta$-function does not obey these conditions; however, the ''legitimacy'' of this function can be established by advanced Fourier Analysis.

<a name="5.5"></a>
## 5.5 Fourier Transform Pairs I: The Boxcar and the Sinc

Let's now examine a number of canonical Fourier transforms that will be very useful for this course.

<b>Example 1a: boxcar (time domain)</b>

**Q:** Find the Fourier Transform of $x(t)$, which is a box-car pulse $\Pi_{T_0}$ of the form:

$$
x(t) = \Pi_{T_0}(t)
=
\left\{
\begin{array}{ll}
1, & -T_0<t<T_0\\
0, & \mathrm{otherwise}
\end{array}
\right.  \tag{12}
$$

**A:** Note that the signal is of finite length and corresponds to one period of the boxcar example examined in Module 4. Here $\Pi_{T_0}$ has <i>half-width</i> $T_0$ (total duration $2T_0$), the convention used throughout these notes.

$$
\begin{eqnarray}
\widehat{X}(\omega) = \int_{-\infty}^{\infty} x(t)\,\mathrm{e}^{-i\omega t}\,\mathrm{d}t \tag{13} \\
= \int_{-T_0}^{T_0} \mathrm{e}^{-i\omega t}\,\mathrm{d}t \tag{14}\\
= \left.\frac{\mathrm{e}^{-i\omega t}}{-i\omega}\right|_{-T_0}^{T_0} \tag{15} \\
= \frac{\mathrm{e}^{-i\omega T_0}-\mathrm{e}^{i\omega T_0}}{-i\omega} \tag{16}\\
= \frac{\mathrm{e}^{i\omega T_0}-\mathrm{e}^{-i\omega T_0}}{i\omega} \tag{17}\\
= \frac{2\mathrm{sin}(\omega T_0)}{\omega} \tag{18}\\
= \frac{2T_0\mathrm{sin}(\omega T_0)}{\omega T_0} \tag{19} \\
= 2T_0 \,\mathrm{sinc} \left(\frac{\omega T_0}{\pi}\right) \tag{20}
\end{eqnarray}
$$

where in the last step we have used the <b>normalized sinc function</b>, $\mathrm{sinc}(x) = \sin(\pi x)/(\pi x)$, which is the definition used throughout these notes and by <code>np.sinc</code>. Note that for $T_0=1/2$, and because $\omega=2\pi f$, we obtain:

$$ \frac{2\mathrm{sin}(\omega T_0)}{\omega} = \frac{2\mathrm{sin}(2 \pi f/2)}{2 \pi f} = \frac{\mathrm{sin}(\pi f)}{\pi f} = \mathrm{sinc} (f) \tag{21}$$

which is the **sinc** function that is very important in digital sampling and reconstruction.

<b>Example 1b: boxcar (frequency domain)</b>

**Q:** Find the inverse Fourier transform of $\widehat{X}(\omega)$ which is a rectangular box-car pulse $\widehat{\Pi}(\omega)$ of the form:

$$
\widehat{X}(\omega) = 
\widehat{\Pi}(\omega)
=\left\{
\begin{array}{ll}
1, & -\omega_0<\omega<\omega_0\\
0, & \mathrm{otherwise}
\end{array}
\right. \tag{22}
$$

**A:** From above we get:

$$
\begin{eqnarray}
x(t) = \frac{1}{2\pi}\int_{-\infty}^{\infty}\widehat{X}(\omega)\,\mathrm{e}^{i\omega t}\, \mathrm{d}\omega \tag{23} \\
= \frac{1}{2\pi}\int_{-\omega_0}^{\omega_0} \mathrm{e}^{i\omega t}\,\mathrm{d}\omega \tag{24}\\
= \left.\frac{\mathrm{e}^{i\omega t}}{i2\pi t} \right|_{-\omega_0}^{\omega_0} \tag{25}\\
= \frac{\mathrm{e}^{i\omega_0 t} - \mathrm{e}^{-i\omega_0 t}}{i2\pi t} \tag{26}\\
= \frac{\mathrm{sin}(\omega_0 t)}{\pi t}  \tag{27}\\ 
\end{eqnarray}
$$

Similarly, if $\omega_0=2\pi f_0$ and $f_0=1/2$ then we get

$$x(t) = \frac{\mathrm{sin}(\omega_0 t)}{\pi t} = \frac{\mathrm{sin}(2 \pi f_0 t)}{\pi t} =\frac{\mathrm{sin}(\pi t)}{\pi t} = \mathrm{sinc}(t) \tag{28}$$

Note that these two demonstrate the **duality property** of the Fourier Transform: The (inverse) Fourier transform of a boxcar function is a (scaled) sinc function! The only differences are the normalization factor of $\frac{1}{2\pi}$ and that the time domain has a $T_0$ while the Fourier domain has $\omega_0$.

Figure 5.6 checks Example 1a numerically: the transform computed with <code>np.fft.fft</code> (scaled by $\Delta t$) falls on the analytic
curve, with its peak $\widehat X(0)=2T_0$ (the area under the boxcar) and its first zeros at $\omega=\pm\pi/T_0$.

In [ ]:
## . . Boxcar of half-width T0 and its transform: analytic (equation 11) vs. the FFT approximation dt*fft(x)
T0, dt = 0.5, 1e-3
N = 64000                                   # 64 s record
t = (np.arange(N) - N//2)*dt
x = np.where(np.isclose(np.abs(t), T0), 0.5, (np.abs(t) < T0).astype(float))
X = dt*np.fft.fftshift(np.fft.fft(np.fft.ifftshift(x)))     # ifftshift puts t = 0 at index 0
w = 2*np.pi*np.fft.fftshift(np.fft.fftfreq(N, dt))          # angular frequency (rad/s)
wa = np.linspace(-60, 60, 2001)

fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.4))
axa.plot(t, x, color=DSP_COLORS['time_domain'])
axa.set_xlim(-1.5, 1.5); axa.set_ylim(-0.1, 1.15)
axa.set_xlabel('time $t$ [s]'); axa.set_ylabel(r'$x(t)=\Pi_{T_0}(t)$'); axa.set_title(r'(a) Boxcar, $T_0=0.5$ s', fontsize=12)
axb.plot(wa, 2*T0*np.sinc(wa*T0/np.pi), color=DSP_COLORS['freq_domain'], label=r'analytic $2T_0\,\mathrm{sinc}(\omega T_0/\pi)$')
m = (np.abs(w) < 60)
axb.plot(w[m][::12], X[m][::12].real, 'o', ms=4, color='k', label=r'FFT: $\Delta t\cdot$fft$(x)$')
axb.axvline(np.pi/T0, ls=':', color=DSP_COLORS['neutral']); axb.axvline(-np.pi/T0, ls=':', color=DSP_COLORS['neutral'])
axb.text(np.pi/T0 + 1, 0.8, r'$\omega=\pi/T_0$', fontsize=11)
axb.set_xlim(-60, 60); axb.set_xlabel(r'angular frequency $\omega$ [rad/s]'); axb.set_ylabel(r'$\widehat{X}(\omega)$')
axb.set_title('(b) Its Fourier transform', fontsize=12); axb.legend(loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

<b>Figure 5.6.</b> (a) A boxcar of half-width $T_0=0.5$ s and (b) its Fourier transform: the analytic $2T_0\,\mathrm{sinc}(\omega T_0/\pi)$ (red) and the FFT approximation $\Delta t\cdot$fft$(x)$ with $\Delta t=1$ ms over a 64 s record (dots; every 12th point), which agree to better than 0.01. The dotted lines mark the first zeros at $\omega=\pm\pi/T_0=\pm 6.28$ rad/s. <i>(Python-generated.)</i>

<div class="alert alert-warning">
<b>Common misconception:</b> "sinc$(x)$ means $\sin x/x$." Many books define sinc that way. <code>np.sinc(x)</code>
is $\sin(\pi x)/(\pi x)$, the normalized sinc used here, so the transform of a boxcar is <code>2*T0*np.sinc(w*T0/np.pi)</code>, not
<code>2*T0*np.sinc(w*T0)</code>. Check which definition a formula assumes before you code it.
</div>

<a name="5.6"></a>
## 5.6 Properties of the Fourier Transform

There are a number of key properties of the Fourier transform that are well worth knowing.  Assume that $f(t)$, $g(t)$ and $h(t)$ are **integrable functions** obeying, 

$$ \int_{-\infty}^{\infty} \left|f(t)\right| \mathrm{d}t \lt \infty, \quad \quad \int_{-\infty}^{\infty} \left|g(t)\right| \mathrm{d}t \lt \infty, \quad {\rm and} \quad \int_{-\infty}^{\infty} \left|h(t)\right| \mathrm{d}t \lt \infty \tag{29} $$

and have Fourier transforms $\widehat{F}(\omega)$, $\widehat{G}(\omega)$ and $\widehat{H}(\omega)$, respectively.  Given these requirements, the following are six key properties: Linearity, Translation, Modulation, Time scaling, Conjugation and Integration.

#### Linearity

For any complex numbers $a$ and $b$, if $h(t)=a f(t)+b g(t)$ then

$$ \widehat{H}(\omega) = a \widehat{F}(\omega)+ b \widehat{G}(\omega) \tag{30} $$

#### Translation / time shifting

For any real number $t_0$, if $h(t)=f(t-t_0)$ then 


$$ \widehat{H}(\omega)  = \mathrm{e}^{-i \omega t_0 }\,\widehat{F}(\omega) \tag{31} $$

#### Modulation / frequency shifting

For any real number $\omega_0$ if $h(t) = \mathrm{e}^{i\omega_0t} f(t)$ then

$$  \widehat{H}(\omega) = \widehat{F}(\omega-\omega_0) \tag{32}$$

<div class="alert alert-info">
<b>Synthesis &mdash; multiplying in time moves things in frequency.</b> Multiplying $f(t)$ by $e^{i\omega_0t}$ slides its whole spectrum to
$\omega_0$ (equation 32); multiplying by $\cos\omega_0t$ makes two half-height copies at $\pm\omega_0$ (Euler, Module 3). Read backward, this
explains the beat of Figure 5.3: for equal amplitudes, $\cos a+\cos b=2\cos\tfrac{a+b}{2}\cos\tfrac{a-b}{2}$ (PHGN 200), a 4 Hz tone multiplied
by a slow 0.05 Hz envelope whose magnitude swells every 10 s. This is the simplest case of a duality between products in one domain and
convolutions in the other, which the course returns to repeatedly.
</div>

#### Time Scaling (and Time Reversal)

For any non-zero real constant $a$, if $h(t) = f(at)$ then

$$ \widehat{H}(\omega) = \frac{1}{|a|}\widehat{F}(\omega/a) \tag{33}$$

One key thing to remember is that when the time axis is compressed (i.e., $|a|>1$, so $f(at)$ is narrower than $f(t)$), the frequency axis is stretched; when $|a|<1$ the spectrum is compressed (and vice versa).  Note that the case $a=-1$ leads to the **time-reversal** property, which states if $h(t)=f(-t)$ then $\widehat{H}(\omega) = \widehat{F}(-\omega)$.

The animation in Figure 5.7 narrows a boxcar tenfold, from $T_0=1$ s to $T_0=0.1$ s, and draws its transform from Example 1a
at each step. Watch the product in the title: it stays at $\pi$ while the boxcar narrows and the main lobe of the spectrum widens by the same
factor of ten. The peak height $2T_0$ falls because the area under $x(t)$, which equals $\widehat X(0)$ (equation 35), falls.

In [ ]:
C = DSP_COLORS
T0s = np.geomspace(1.0, 0.1, 30)            # boxcar half-width shrinks tenfold over 30 frames
t = np.linspace(-1.5, 1.5, 3001)
w = np.linspace(-80, 80, 3201)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(10, 3.7))
lx, = axa.plot([], [], color=C['time_domain'])
lX, = axb.plot([], [], color=C['freq_domain'])
n1 = axb.axvline(0, ls=':', color=C['neutral']); n2 = axb.axvline(0, ls=':', color=C['neutral'])
axa.set_xlim(-1.5, 1.5); axa.set_ylim(-0.1, 1.15); axa.set_xlabel('time $t$ [s]'); axa.set_ylabel('$x(t)$')
axa.set_title(r'(a) Boxcar of half-width $T_0$', fontsize=12)
axb.set_xlim(-80, 80); axb.set_ylim(-0.5, 2.15); axb.set_xlabel(r'$\omega$ [rad/s]'); axb.set_ylabel(r'$\widehat{X}(\omega)$')
axb.set_title(r'(b) $\widehat{X}(\omega)=2T_0\,\mathrm{sinc}(\omega T_0/\pi)$', fontsize=12)
title = fig.suptitle('', fontsize=11)

def update(k):
    T0 = T0s[k]
    lx.set_data(t, (np.abs(t) < T0).astype(float))
    lX.set_data(w, 2*T0*np.sinc(w*T0/np.pi))
    wn = np.pi/T0
    n1.set_xdata([wn, wn]); n2.set_xdata([-wn, -wn])
    title.set_text(f'$T_0$ = {T0:.3f} s,   first null $\\omega=\\pi/T_0$ = {wn:.1f} rad/s,   '
                   f'$T_0\\times\\omega_\\mathrm{{null}}$ = {T0*wn:.4f} = $\\pi$')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=len(T0s), interval=300, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 5.7.</b> The scaling theorem in action, 30 frames. (a) A boxcar whose half-width $T_0$ shrinks geometrically from 1 s to 0.1 s; (b) its transform $2T_0\,\mathrm{sinc}(\omega T_0/\pi)$, with the first zeros at $\omega=\pm\pi/T_0$ (dotted). Half-width times first-null frequency is always $\pi$. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; you cannot be narrow in both domains.</b> The scaling theorem (equation 33) makes Module 4's squeeze/stretch argument exact:
compress a signal by $a$ and its spectrum stretches by $a$ while its height drops by $1/|a|$. For the boxcar, half-width times first-null
frequency is always $\pi$ (Figure 5.7); for the Gaussian (Figure 5.8) the product of the 1/e half-widths is always 2. A short seismic source
gives a broad band; a long one gives a narrow band (Microquestion 5.2).
</div>

#### Conjugation

If $h(t)=\overline{f(t)}$ then,

$$\widehat{H}(\omega) = \overline{\widehat{F}(-\omega)} \tag{34}$$

Note that if $f(t)$ is real then $\widehat{F}(-\omega)=\overline{\hat{F}(\omega)}$.  Similarly, if it is purely imaginary then $\widehat{F}(-\omega)=-\overline{\widehat{F}(\omega)}$.

#### Integration

Substituting $\omega=0$ in the definition of the Fourier Transform, we obtain

$$ \widehat{F}(0)=\int_{-\infty}^{\infty} f(t) \,\mathrm{d}t \tag{35} $$

which is equivalent to recovering the [DC Component](https://en.wikipedia.org/wiki/DC_bias) of the Fourier spectrum.

<a name="5.6.1"></a>
### 5.6.1 Proofs of three properties

Prove the following Fourier transform properties. Try each one before opening the proof.

<div class="alert alert-info">
<b>Proof:</b> <b>(1) Time translation.</b> Show if $h(t)=f(t-t_0)$ then $\widehat{H}(\omega) = e^{-i\omega t_0}\widehat{F}(\omega)$
   
   $$
   \begin{eqnarray}
   \widehat{H}(\omega) &=& \int_{-\infty}^{\infty} f(t-t_0) e^{-i\omega t} \mathrm{d}t \tag{36} \\
                        &=& \int_{-\infty}^{\infty} f(t-t_0) e^{-i\omega t} e^{-i\omega t_0} e^{i\omega t_0} \mathrm{d}t \tag{37}\\
                        &=& \int_{-\infty}^{\infty} f(t-t_0) e^{-i\omega (t-t_0)} e^{-i\omega t_0} \mathrm{d}t \tag{38}\\                        
   \end{eqnarray}
   $$

Now, since (1) $t_0$ is a constant, then the effective differential is given by $\mathrm{d}(t-t_0)=\mathrm{d}t$; and (2) $e^{-i\omega t_0}$ is independent of $t$, we can write:

   $$
   \begin{eqnarray}
\widehat{H}(\omega)&=& e^{-i\omega t_0}\int_{-\infty}^{\infty} f(t-t_0) e^{-i\omega (t-t_0)} \mathrm{d}(t-t_0) \tag{39} \\
   \end{eqnarray}
   $$

Defining $t^\prime=t-t_0$, we write:

   $$
   \begin{eqnarray}
\widehat{H}(\omega)&=& e^{-i\omega t_0}\int_{-\infty}^{\infty} f(t^\prime) e^{-i\omega t^\prime} \mathrm{d}t^\prime \tag{40}\\
&=& e^{-i\omega t_0} \widehat{F}(\omega) \tag{41}\\
\end{eqnarray}
   $$
</div>

<div class="alert alert-info">
<b>Proof:</b> <b>(2) Frequency modulation.</b> Show if $h(t) = \mathrm{e}^{i\omega_0t} f(t)$ then $\widehat{H}(\omega) = \widehat{F}(\omega-\omega_0)$
  
   $$
   \begin{eqnarray}
\widehat{H}(\omega)&=& \int_{-\infty}^{\infty} f(t) e^{i\omega_0 t} e^{-i\omega t} \mathrm{d}t \tag{42}\\
&=& \int_{-\infty}^{\infty} f(t)  e^{-i(\omega-\omega_0) t} \mathrm{d}t \tag{43}\\
\end{eqnarray}
   $$  

Let $\omega^\prime = \omega-\omega_0$. Then we can write:

   $$
   \begin{eqnarray}
\widehat{H}(\omega) &=& \int_{-\infty}^{\infty} f(t)  e^{-i\omega^\prime t} \mathrm{d}t \tag{44}\\
&=& \widehat{F}(\omega^\prime) \tag{45}\\
&=& \widehat{F}(\omega-\omega_0) \tag{46} 
\end{eqnarray}
   $$
</div>

<div class="alert alert-info">
<b>Proof:</b> <b>(3) Time scaling.</b> If $h(t) = f(at)$ (for real and constant $a \gt 0$) then $ \widehat{H}(\omega) = \frac{1}{a}\widehat{F}(\omega/a)$

   $$
   \begin{eqnarray}
\widehat{H}(\omega)&=& \int_{-\infty}^{\infty} f(at) e^{-i\omega t} \mathrm{d}t \tag{47} \\
\end{eqnarray}
   $$  
   
Define $t^\prime=at$ such that $t=t^\prime/a$ and $\mathrm{d}t = \mathrm{d}t^\prime/a$. Then,

   $$
   \begin{eqnarray}
\widehat{H}(\omega)&=& \int_{-\infty}^{\infty} f(t^\prime) e^{-i\omega t^\prime/a} \mathrm{d}t^\prime/a \tag{48}\\
\end{eqnarray}
   $$  
   
Defining $\omega^\prime = \omega/a$ and moving constant $a$ outside integral:

   $$
   \begin{eqnarray}
\widehat{H}(\omega)&=& \frac{1}{a} \int_{-\infty}^{\infty} f(t^\prime) e^{-i\omega^\prime t^\prime} \mathrm{d}t^\prime \tag{49}\\
&=& \frac{1}{a} \widehat{F}(\omega^\prime) = \frac{1}{a} \widehat{F}(\omega/a) \tag{50}
\end{eqnarray}
   $$  

Note that one can do this for $a < 0$ to establish the absolute value in equation (33).
</div>

<a name="5.7"></a>
## 5.7 Fourier Transform Pairs II: The Delta Function, Cosine and Sine

There are several key functions that keep appearing through this course (and many others in geophysics!): Dirac delta function, Gaussian, Exponential Decay and the Dirac Comb (also known as the Pulse train or Shah Function).  It is very helpful to remember what the relationship between these functions are in the conjugate domains (e.g., $t$ and $\omega$).

The Dirac delta function, $\delta(t)$, has the following characteristics:

$$ \delta(t) = 0 \quad \mathrm{for} \quad t\neq 0  \tag{51} $$

and 

$$ \int_{-\infty}^{\infty}\delta(t) \, \mathrm{d}t  =  1 \tag{52} $$

Interestingly, we are not defining the value of $\delta(t)$ at $t=0$. Rather we are defining it in terms of an integral.  Note also that this holds true for any delta function defined at $t=a$.  For example,

$$ \int_{-\infty}^{\infty}\delta(t-a) \, \mathrm{d}t  =  1  \tag{53} $$

The other key property studied here is:

$$ f(t)\,\delta(t-t_0)  =  f(t_0)\,\delta(t-t_0) \tag{54} $$

where $f(t)$ is a continuous-time signal.  Equations (51) and (52) indicate that $\delta(t)$ has a very large value or *impulse* at $t=0$.  That is, $\delta(t)$ is not really defined at $t=0$. (Let's leave this alone for the moment.)  Equation (54) is known as the **sifting property** because it "sifts" through all possible answers and only "selects" the one that exactly makes the argument go to zero.

Thus, we could write the following Fourier Transform of $\delta(t)$:

$$ \mathcal{F}[\delta(t)] = \int_{-\infty}^{\infty}\delta(t) \, e^{-i\omega t} \mathrm{d}t = \int_{-\infty}^{\infty}\delta(t) \, e^{-i\omega 0} \mathrm{d}t = \int_{-\infty}^{\infty}\delta(t) \mathrm{d}t = 1 \tag{55} $$


By use of the shift and scaling theorems, one can show that the following functions are Fourier pairs:

$$
\begin{eqnarray}
\delta (t-a) &\Leftrightarrow & \mathrm{e}^{-i\omega a} \tag{56}\\
\delta (mt-a)&\Leftrightarrow & \frac{1}{|m|}\mathrm{e}^{-i\omega a/m}\tag{57}
\end{eqnarray}  
$$

where the symbol $\Leftrightarrow$ indicates a Fourier transform pair.  The first one can be demonstrated by the following:

$$
\begin{eqnarray}
\mathcal{F}[\delta(t-a)]& =& \int_{-\infty}^{\infty} \delta (t-a) e^{-i\omega t} \mathrm{d}t \tag{58} \\
& =& \int_{-\infty}^{\infty} \delta (t-a) e^{-i\omega a} \mathrm{d}t \tag{59} \\
 &=& e^{-i\omega a} \int_{-\infty}^{\infty} \delta (t-a) \mathrm{d}t \tag{60}\\
 &=& e^{-i\omega a}  \tag{61}
\end{eqnarray}
$$

where we have invoked the sifting property to go from equation (58) to (59), and have used the property in equation (53) to go from equation (60) to (61).

The following pairs of $\delta$-functions are useful to remember:

$$ 
\begin{eqnarray}
\frac{ \delta(t+a) + \delta(t-a)}{2} &\Leftrightarrow& \mathrm{cos}\,(a\omega) \tag{62}\\
\frac{ \delta(t+a) - \delta(t-a)}{2i} &\Leftrightarrow&\mathrm{sin}\,(a\omega) \tag{63}\\
\mathrm{cos}\,(at) &\Leftrightarrow& \pi \left[\delta(\omega-a) + \delta(\omega+a)\right] \tag{64} \\
 \mathrm{sin}\,(at) &\Leftrightarrow& \frac{\pi}{i} \left[\delta(\omega-a) - \delta(\omega+a)\right]  
 \tag{65}
\end{eqnarray}
$$

Note also that the Fourier transform of the **real and even** $\cos at$ function is purely **real and even**, whereas the Fourier transform of the **real odd** $\sin at$ function is purely **imaginary and odd**.

<a name="5.8"></a>
## 5.8 Symmetry Properties

The symmetry properties of the Fourier transformation are described in Table 5.1 below.
<center>
    
|Property | $x(t)=x_r(t)+ix_i(t)$  | $\hat{X}(\omega)=\hat{X}_r(\omega)+i\hat{X}_i (\omega)$ |
|:-:|:----------------------:|:--------------------------:|
| 1  | Real $x(t)=x_r(t)$           | Even $\hat{X}_r(\omega)$, Odd $\hat{X}_i(\omega)$ |
| 2  | Real and Even, $x(-t)= x(t)$ |  Real and Even $\hat{X}_r(\omega)$ |
| 3  | Real and Odd, $x(-t)=-x(t)$ |  Imaginary and Odd $\hat{X}_i(\omega)$ |
| 4  | Imaginary $x(t)=x_i(t)$ | Odd $\hat{X}_r(\omega)$, Even $\hat{X}_i(\omega)$ |
| 5  | Imaginary and Even, $x(-t)=x(t)$ | Imaginary and Even $\hat{X}_i(\omega)$|
| 6  | Imaginary and Odd, $x(-t) = - x(t)$ | Real and Odd $\hat{X}_r(\omega)$|
 </center>
<b>Table 5.1.</b> Symmetry properties of the Fourier transform.

<b>Questions about symmetry</b>

Use the $\mathrm{cos}$ and $\mathrm{sin}$ Fourier Transform pairs to demonstrate the six symmetry properties discussed above.
    
**(1) Real and even**

An example of real and even would be $f(t)=\mathrm{cos}(at)$.  Using the Fourier Transform pairs above we can write:

$$\mathcal{F}[f(t)] = \pi \left[\delta(\omega-a) + \delta(\omega+a)\right] \tag{66} $$

for which there are only even real terms.

**(2) Real and odd**

An example of real and odd would be $f(t)=\mathrm{sin}(at)$.  Using the Fourier Transform pairs above we can write:

$$\mathcal{F}[f(t)] = \frac{\pi}{i} \left[\delta(\omega-a) - \delta(\omega+a)\right]  \tag{67} $$

for which there are only odd imaginary terms.

**(3) Real, but neither even nor odd**
    
An example of real but neither even nor odd would be $f(t)=\mathrm{cos}(at) + \mathrm{sin}(at)$.  Using the Fourier Transform pairs above we can write:

$$\mathcal{F}[f(t)] = 
       \pi     \left[\delta(\omega-a) + \delta(\omega+a)\right] 
+\frac{\pi}{i} \left[\delta(\omega-a) - \delta(\omega+a)\right]  \tag{68} 
$$

for which the real terms are even and the imaginary terms are odd.
    
**(4) Imaginary and even**

An example of imaginary and even would be $f(t)=i\mathrm{cos}(at)$.  Using the Fourier Transform pairs above we can write:

$$\mathcal{F}[f(t)] = i\pi \left[\delta(\omega-a) + \delta(\omega+a)\right]  \tag{69} $$

for which there are only even imaginary terms.

**(5) Imaginary and odd**

An example of imaginary and odd would be $f(t)=i\mathrm{sin}(at)$.  Using the Fourier Transform pairs above we can write:

$$\mathcal{F}[f(t)] = \pi \left[\delta(\omega-a) - \delta(\omega+a)\right]  \tag{70} $$

for which there are only odd real terms.

**(6) Imaginary, but neither even nor odd**

An example of imaginary but neither even nor odd would be $f(t)=i\left[\mathrm{cos}(at) + \mathrm{sin}(at)\right]$.  Using the Fourier Transform pairs above we can write:

$$\mathcal{F}[f(t)] = 
       i\pi     \left[\delta(\omega-a) + \delta(\omega+a)\right] 
+\pi \left[\delta(\omega-a) - \delta(\omega+a)\right]  \tag{71} 
$$

for which the real terms are odd and the imaginary terms are even.

<a name="5.9"></a>
## 5.9 Fourier Transform Pairs III: Gaussian, One-Sided Exponential and Dirac Comb

<b>The Gaussian.</b> Suppose that $g(t)=\mathrm{e}^{-(t-m)^2/a^2}$, where $a$ defines the width  and $m$ the center of the Gaussian function.  The Fourier transform is given by

$$ \widehat{G}(\omega) = \int_{-\infty}^{\infty}\mathrm{e}^{-(t-m)^2/a^2} \mathrm{e}^{-i\omega t} \mathrm{d}t \tag{72} $$

which can be solved by a completing-the-square transformation to yield

$$ \widehat{G}(\omega) = a\sqrt{\pi}\,\mathrm{e}^{-im\omega}\,\mathrm{e}^{-a^2\omega^2/4} \tag{73} $$

Importantly for many applications, the Fourier Transform of a Gaussian is also a Gaussian! In addition, the wider the original Gaussian, the narrower will be its Fourier pair (sound familiar?).  Thus, we may write symbolically the following:

$$ g\left(\frac{t}{a}\right) \Leftrightarrow |a|\,\widehat{G}(a\omega) \tag{74} $$

Finally, a Gaussian function is sometimes used in Fourier theory to establish a rigorous basis of the $\delta$-function.  To appreciate why, examine the asymptotic behavior of the following function:

$$\lim_{a\rightarrow 0} \frac{1}{a\sqrt{\pi}}\mathrm{e}^{-t^2/a^2} \approx \delta (t) \tag{75}$$

and its transform tends to the transform of $\delta(t)$ (equation 55): 

$$\lim_{a\rightarrow 0} \mathcal{F}\left[\frac{1}{a\sqrt{\pi}}\mathrm{e}^{-t^2/a^2}\right] = \lim_{a\rightarrow 0} \mathrm{e}^{-a^2\omega^2/4} = 1 \tag{76}$$

Formally establishing the Gaussian as an appropriate test function requires going deeper into Fourier analysis.  (See references below.)

In [ ]:
C = DSP_COLORS
t = np.linspace(-0.5, 0.5, 2001)
w = np.linspace(-100, 100, 2001)
cols = [C['time_domain'], C['impulse_resp'], C['freq_domain']]
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.4))
for a, col in zip([0.05, 0.1, 0.2], cols):
    axa.plot(t, np.exp(-t**2/a**2), color=col, label=f'$a$ = {a} s')
    axb.plot(w, a*np.sqrt(np.pi)*np.exp(-a**2*w**2/4), color=col, label=f'$a$ = {a} s: 1/e at $\\omega=2/a$ = {2/a:.0f} rad/s')
axa.set_xlabel('time $t$ [s]'); axa.set_ylabel(r'$g(t)=e^{-t^2/a^2}$'); axa.set_title('(a) Gaussians of three widths', fontsize=12)
axa.legend(fontsize=9); axa.set_xlim(-0.5, 0.5)
axb.set_xlabel(r'$\omega$ [rad/s]'); axb.set_ylabel(r'$|\widehat{G}(\omega)|=a\sqrt{\pi}\,e^{-a^2\omega^2/4}$')
axb.set_title('(b) Their transforms', fontsize=12); axb.legend(fontsize=9, loc='upper left'); axb.set_xlim(-100, 100)
axb.set_ylim(0, 0.48)
plt.tight_layout(); plt.show()

<b>Figure 5.8.</b> Gaussians and their transforms, equation (73) with $m=0$. (a) $g(t)=e^{-t^2/a^2}$ for $a=0.05$, 0.1 and 0.2 s. (b) $|\widehat G(\omega)|$, which falls to 1/e of its peak at $\omega=2/a$. The narrowest pulse has the widest spectrum, and the product of the 1/e half-widths, $a\times 2/a$, is 2 for every $a$. <i>(Python-generated.)</i>

<b>The one-sided exponential.</b> The **exponential decay function** is commonly given by $d(t) = u(t)\,\mathrm{e}^{-at}$ where $a>0$ and $u(t)$ is the **unit-step** (or **Heaviside**) function having the form:

$$
u(t) = \left\{ 
\begin{array}{ll}
1, & t\ge0 \\
0, & t<0 
\end{array}
\right. \tag{77}
$$

Note that the sudden change from 0 to 1 at $t=0$ means that $u(0)$ is not that well defined (which we will again leave until later).  The exponential decay function may then be written as:

$$ d(t) = \left\{
\begin{array}{ll}
\mathrm{e}^{-at}, & t\ge 0 \\
0, & t \lt 0
\end{array}
\right. \tag{78}
$$

The Fourier transform of this function is given by:

$$ 
\begin{eqnarray}
\hat{D}(\omega) & = & \int_{0}^{\infty} \,\mathrm{e}^{-at}\,\mathrm{e}^{-i \omega t} \,\mathrm{d}t\\
\, & = & -\left[
\frac{
\mathrm{e}^{-(a+i \omega)t}
}{a+i\omega}
\right]^{\infty}_{0}
= \frac{1}{a+i\omega}=
\frac{a-i\omega}{a^2+\omega^2}
\end{eqnarray} \tag{79}
$$

In [ ]:
C = DSP_COLORS
a = 2.0
t = np.linspace(-1, 4, 2001)
w = np.linspace(-20, 20, 2001)
fig, axs = plt.subplots(1, 3, figsize=(14, 4.2))
axs[0].plot(t, np.where(t >= 0, np.exp(-a*t), 0.0), color=C['time_domain'])
axs[0].set_xlabel('time $t$ [s]'); axs[0].set_ylabel('$d(t)$'); axs[0].set_title(r'(a) $d(t)=u(t)\,e^{-at}$, $a=2$ s$^{-1}$', fontsize=12)
axs[1].plot(w, 1/np.sqrt(a**2 + w**2), color=C['freq_domain'])
axs[1].set_xlabel(r'$\omega$ [rad/s]'); axs[1].set_ylabel(r'$|\widehat{D}(\omega)|$'); axs[1].set_title(r'(b) Magnitude $1/\sqrt{a^2+\omega^2}$', fontsize=12)
axs[2].plot(w, -np.degrees(np.arctan(w/a)), color=C['phase'])
axs[2].plot([a], [-45], 'o', color='k'); axs[2].annotate(r'$-45^\circ$ at $\omega=a$', (a, -45), (a + 3, -20), fontsize=10,
                                                       arrowprops=dict(arrowstyle='->', color='k'))
axs[2].set_xlabel(r'$\omega$ [rad/s]'); axs[2].set_ylabel('phase [degrees]'); axs[2].set_title(r'(c) Phase $-\arctan(\omega/a)$', fontsize=12)
axs[2].set_ylim(-95, 95)
plt.tight_layout(); plt.show()

<b>Figure 5.9.</b> The one-sided exponential $d(t)=u(t)e^{-at}$ with $a=2$ s$^{-1}$ and its transform $1/(a+i\omega)$, equation (79): (a) $d(t)$; (b) the magnitude $1/\sqrt{a^2+\omega^2}$; (c) the phase $-\arctan(\omega/a)$, which passes $-45^\circ$ at $\omega=a$ and tends to $\mp 90^\circ$. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; the Laplace transform from MATH 225.</b> For a signal that is zero for $t<0$, the Fourier transform is the MATH 225 Laplace
transform $\mathcal L\{x\}(s)=\int_0^\infty x(t)e^{-st}dt$ evaluated on the imaginary axis, $s=i\omega$. The table entry $\mathcal L\{e^{-at}\}=1/(s+a)$
becomes $1/(a+i\omega)$, which is equation (79). The Fourier version needs the integral to converge with $s=i\omega$, which is why $a>0$ is required.
</div>

<b>The Dirac comb.</b> A Dirac comb (also known as a <b>pulse train</b> or <b>Shah function</b>) is an infinite set of equally spaced
$\delta$-functions separated by period $T$. (It is usually denoted by the Cyrillic letter ''shah'', written below as $\mathrm{III}_T(t)$):

$$ \mathrm{III}_T(t) = \sum_{n=-\infty}^{\infty}\delta(t-nT) \tag{80}$$

This function is important because it allows Fourier series to be included in Fourier transform theory, and it represents
<b>digital sampling</b>. Its transform follows from two results we already have. The comb is periodic with period $T$, so it has a
Fourier series (Module 4) whose coefficients are all equal: $C_k=\frac{1}{T}\int_{-T/2}^{T/2}\delta(t)\,e^{-ik\omega_0t}\,dt=\frac{1}{T}$, with
$\omega_0=2\pi/T$. Each term $e^{ik\omega_0t}$ transforms to $2\pi\,\delta(\omega-k\omega_0)$ (equation 64 with the modulation property). Hence

$$ \mathrm{III}_T(t)=\frac{1}{T}\sum_{k=-\infty}^{\infty}e^{ik\omega_0 t} \;\Leftrightarrow\; \frac{2\pi}{T}\sum_{k=-\infty}^{\infty}\delta\!\left(\omega-\frac{2\pi k}{T}\right)=\frac{2\pi}{T}\,\mathrm{III}_{2\pi/T}(\omega) \tag{81}$$

The Fourier transform of a Dirac comb is also a Dirac comb: spikes every $T$ in time become spikes every $2\pi/T$ in frequency. Figure 5.10 shows
the discrete version of this pair.

In [ ]:
C = DSP_COLORS
N, P = 400, 8                       # N samples, one spike every P samples
x = np.zeros(N); x[::P] = 1.0
X = np.fft.fft(x)/N
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.2))
ml, sl, bl = axa.stem(np.arange(64), x[:64])
plt.setp(ml, color=C['sampled'], ms=4); plt.setp(sl, color=C['sampled']); plt.setp(bl, color='k', lw=1)
axa.set_xlabel('sample $n$'); axa.set_ylabel('$x[n]$'); axa.set_title('(a) Comb: a spike every $P=8$ samples (first 64 shown)', fontsize=11)
k = np.nonzero(np.abs(X) > 1e-9)[0]                 # only the nonzero bins
ml, sl, bl = axb.stem(k, np.abs(X[k])); axb.axhline(0, color='k', lw=1)
plt.setp(ml, color=C['freq_domain'], ms=4); plt.setp(sl, color=C['freq_domain']); plt.setp(bl, color='k', lw=1)
axb.annotate('', xy=(100, 0.135), xytext=(50, 0.135), arrowprops=dict(arrowstyle='<->', color='k'))
axb.text(75, 0.139, '$N/P=50$ bins', ha='center', fontsize=10)
axb.set_ylim(0, 0.155); axb.set_xlim(-5, N + 5)
axb.set_xlabel('frequency index $k$'); axb.set_ylabel(r'$|X[k]|/N$'); axb.set_title('(b) Its DFT: a comb of height $1/P$', fontsize=11)
plt.tight_layout(); plt.show()

<b>Figure 5.10.</b> The discrete version of equation (81). (a) A comb with one spike every $P=8$ samples in an $N=400$-sample record (first 64 samples shown). (b) The magnitude of its DFT divided by $N$: nonzero only at every $N/P=50$th frequency index, with height $1/P=0.125$. A short spacing in time gives a wide spacing in frequency. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; the comb links series and transforms.</b> A signal that repeats with period $T$ has the transform $2\pi\sum_kC_k\,\delta(\omega-k\omega_0)$:
Module 4's discrete line spectrum, written as spikes. The comb itself is the extreme case, with every $C_k=1/T$. Reading the pair the other way,
the DTFT of equation (83) shows that a signal known only at times $nT$ has a spectrum that repeats every $2\pi/T$. Module 10 builds sampling
theory on this pair.
</div>

<b>Table 5.2.</b> Fourier transform pairs from this module ($\widehat X(\omega)=\int x(t)e^{-i\omega t}dt$; $\mathrm{sinc}(x)=\sin(\pi x)/(\pi x)$).

| $x(t)$ | $\widehat X(\omega)$ | Equation |
|:--|:--|:--|
| boxcar $\Pi_{T_0}(t)$, half-width $T_0$ | $2T_0\,\mathrm{sinc}(\omega T_0/\pi)$ | (20) |
| $\sin(\omega_0 t)/(\pi t)$ | boxcar $\widehat\Pi(\omega)$, half-width $\omega_0$ | (27) |
| $\delta(t)$ | $1$ | (55) |
| $\delta(t-a)$ | $e^{-i\omega a}$ | (56) |
| $1$ | $2\pi\,\delta(\omega)$ | (64), $a=0$ |
| $\cos(at)$ | $\pi[\delta(\omega-a)+\delta(\omega+a)]$ | (64) |
| $\sin(at)$ | $\frac{\pi}{i}[\delta(\omega-a)-\delta(\omega+a)]$ | (65) |
| Gaussian $e^{-t^2/a^2}$ | $a\sqrt{\pi}\,e^{-a^2\omega^2/4}$ | (73) |
| $u(t)\,e^{-at}$, $a>0$ | $1/(a+i\omega)$ | (79) |
| comb $\mathrm{III}_T(t)$ | $\frac{2\pi}{T}\,\mathrm{III}_{2\pi/T}(\omega)$ | (81) |

<a name="5.10"></a>
## 5.10 From Continuous to Sampled: The DTFT and the DFT

So far in my numerical examples I have used words like "continuous function".  However, you'll have noticed in the code snippets that these are actually discrete functions that are densely sampled so they appear continuous.  Actually, what I have done is sample the analytically defined (and aperiodic) continuous function with a Dirac Comb:

$$ x_s(t) = \sum_{n=-\infty}^{\infty} x[n] \delta(t-nT) \tag{82}$$

where $x_s(t)$ represents the sampled version of $x(t)$.  Importantly, $t$ is still a continuous variable in this context.  Taking the Fourier Transform of $x_s(t)$ and using the properties of the $\delta$-function allows us to write:

$$ \widehat{X_s}(\omega) = \int_{-\infty}^{\infty} x_s(t)\, \mathrm{e}^{-i\omega t}\,\mathrm{d}t = \int_{-\infty}^{\infty}  \sum_{n=-\infty}^{\infty} x[n] \delta(t-nT) \, \mathrm{e}^{-i\omega t}\,\mathrm{d}t
= \sum_{n=-\infty}^{\infty} x[n]\, \mathrm{e}^{-i\omega nT} \tag{83}
$$

the last term of which is called the [Discrete Time Fourier Transform](https://en.wikipedia.org/wiki/Discrete-time_Fourier_transform) or (DTFT).  Unfortunately, the DTFT is impractical for use in most situations because it involves dealing with **infinite series** of impulses! However, the DTFT is commonly used in **filtering theory** to design and analyze filters of predefined coefficients and finite length in terms of causality, stability, impulse response, etc. (Designing a filter is much different than "designing your data"!) 

In practical applications of examining Fourier spectra, you will mostly be using a finite-time series that encapsulates the (often transient) signals you want to analyze. In this case, you can make an argument that the recorded digitized signal is actually **discrete** and **periodic**, which allows you to use a **discrete Fourier Transform** (or DFT) or more likely a **Fast Fourier Transform** (or FFT) program.  Thus, in the sections below I am using NumPy's FFT (<code>np.fft</code>) to compute the Fourier spectra, not the analytic Fourier transform; Figure 5.6 showed how closely the two agree for a well-sampled transient.

<a name="5.11"></a>
## 5.11 A Geophysical Example: A Seismic Shot Record

Let's look at an example of taking a Fourier Transform of a seismic shot record (i.e. along the vertical time axis). The record is a land shot gather from the Capricorn deep seismic survey in Western Australia, sampled every 4 ms.

In [ ]:
## . . NOTE: Must have read_in_rsf.py utility file in the same directory as your notebook
a = sf.read_rsf_file('data/CAPshot', use_memmap=True)

nt = 400              # . . Number of time samples used (1.6 s)
dt = a.view.dx[0]     # . . Time sampling interval (0.004 s)
nx = a.view.nx[1]     # . . Number of traces (300)
shot = np.array(a.data[:nt, :nx], dtype=float)

## . . A few samples next to the source are enormous (instrument clipping/spikes): zero them
spikes = np.abs(shot) > 1e3*np.median(np.abs(shot))
shot[spikes] = 0.0
tr = sorted(set(np.nonzero(spikes)[1].tolist()))
print(f'zeroed {spikes.sum()} spike samples on traces {tr[0]}-{tr[-1]}')

In [ ]:
## . . (a) gather with each trace scaled to unit RMS; (b) magnitude spectrum of each trace, in dB re its own peak
g = shot/(np.sqrt(np.mean(shot**2, axis=0, keepdims=True)) + 1e-30)
f = np.fft.rfftfreq(nt, dt)
S = np.abs(np.fft.rfft(shot, axis=0))
SdB = 20*np.log10(S/(S.max(axis=0, keepdims=True) + 1e-30) + 1e-12)
spike_traces = sorted(set(np.nonzero(spikes)[1]))
SdB[:, spike_traces] = np.nan          # . . traces that held spikes are left blank (gray) in (b)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.6))
clip = np.percentile(np.abs(g), 98)
axa.imshow(g, cmap='gray', vmin=-clip, vmax=clip, aspect='auto', extent=[0, nx, nt*dt, 0])
axa.set_xlabel('trace number'); axa.set_ylabel('time [s]'); axa.set_title('(a) Shot gather (first 1.6 s)', fontsize=12)
cmap = plt.get_cmap('viridis').copy(); cmap.set_bad('0.75')
im = axb.imshow(SdB, cmap=cmap, vmin=-40, vmax=0, aspect='auto', origin='lower', extent=[0, nx, 0, f[-1]])
axb.set_xlabel('trace number'); axb.set_ylabel('frequency [Hz]'); axb.set_title('(b) Magnitude spectrum of each trace', fontsize=12)
cb = fig.colorbar(im, ax=axb, pad=0.02); cb.set_label('dB relative to the trace peak')
plt.tight_layout(); plt.show()

<b>Figure 5.11.</b> (a) The first 1.6 s of a land seismic shot gather (300 traces), each trace scaled to unit RMS; 892 spike samples on traces 139&ndash;155, next to the source, were set to zero. (b) The magnitude spectrum of each trace along the time axis, in decibels relative to that trace's own peak, from 0 to the Nyquist frequency of 125 Hz; the traces that held spikes are left blank (gray). <i>(Python-generated; data from <a href="https://researchdata.edu.au/l195-capricorn-deep-10ga-cp3/1229896">Research Data Australia</a>.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; reading a shot record in frequency.</b> Each column of Figure 5.11(b) is the spectrum of one trace (Module 3: magnitude and phase of each frequency; only the magnitude is shown). Traces next to the source, dominated by slow surface waves, peak near 14 Hz with most energy between about 13 and 27 Hz (within 6 dB of the peak); distant traces have a broader band, about 13&ndash;62 Hz. Less than 0.5% of any trace's energy lies above 100 Hz: the upper limit is set by the source and by attenuation in the Earth, the anelastic loss of high frequencies discussed in POTE1 (GPGN 328). That band is what any later filtering must respect.
</div>

<a name="5.12"></a>
## 5.12 Instruments and Earthquakes Revisited

At the start of this module we examined a number of different signals. Having read and digested the sections above, you can now look at their Fourier spectra! Figure 5.12 shows the four instruments of Figure 5.1; you can replay them with the players in Section 5.1.

In [ ]:
## . . One-sided magnitude spectra (normalized by the number of samples), with the harmonics k x 523.25 Hz dashed
C5 = 440*2**(3/12)
fig, axs = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for ax, (lab, xx, rate) in zip(axs, [('(a) Piano', piano, srp), ('(b) Flute', flute, srf),
                                     ('(c) Trumpet', trumpet[tstart:], srt), ('(d) Violin', violin, srv)]):
    F = np.abs(np.fft.rfft(xx))/len(xx); fr = np.fft.rfftfreq(len(xx), 1/rate)
    for k in range(1, 10):
        ax.axvline(k*C5, ls='--', lw=0.8, color=DSP_COLORS['neutral'])
    ax.plot(fr, F, color=DSP_COLORS['freq_domain'], lw=0.9)
    ax.set_ylabel('magnitude'); ax.set_title(f'{lab}: strongest peak at {fr[np.argmax(F)]:.1f} Hz', fontsize=12)
    ax.set_ylim(0, 1.1*F.max())
axs[-1].set_xlim(0, 5000); axs[-1].set_xlabel('frequency [Hz]')
plt.tight_layout(); plt.show()

<b>Figure 5.12.</b> One-sided magnitude spectra of the four instruments of Figure 5.1, 0&ndash;5 kHz, with the harmonics $k\times 523.25$ Hz of C5 dashed: (a) piano, (b) flute, (c) trumpet, (d) violin. All four play the same note, so their peaks fall on the same harmonics; they sound different because the harmonics have different strengths. In the trumpet the second harmonic (1047 Hz) is stronger than the fundamental. <i>(Python-generated.)</i>

In [ ]:
F = np.abs(np.fft.rfft(equake.astype(float)))/equake.size
fr = np.fft.rfftfreq(equake.size, 1/sr)
fig, (axa, axb) = plt.subplots(2, 1, figsize=(12, 6))
axa.plot(np.arange(equake.size)/sr, equake, color=DSP_COLORS['time_domain'], lw=0.6)
axa.set_xlabel('time [s]'); axa.set_ylabel('amplitude'); axa.set_title('(a) Earthquake time series', fontsize=12)
axa.set_xlim(0, equake.size/sr)
axb.plot(fr, F, color=DSP_COLORS['freq_domain'], lw=0.8)
axb.set_xlim(0, 300); axb.set_ylim(0, 1.1*F.max())
axb.set_xlabel('frequency [Hz]'); axb.set_ylabel('magnitude'); axb.set_title(f'(b) Magnitude spectrum: peak at {fr[np.argmax(F)]:.1f} Hz', fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 5.13.</b> (a) The earthquake recording of Figure 5.2 and (b) its one-sided magnitude spectrum, 0&ndash;300 Hz, which peaks at 70 Hz. Unlike the instruments, the spectrum is a continuous band with no harmonic lines: the signal does not repeat. (The frequencies are in the time base of the audio file.) <i>(Python-generated.)</i>

<a name="5.13"></a>
## 5.13 Parting Thoughts

It's important to develop a good conceptual understanding of how time-domain signals "look" in the Fourier domain.  This will help you "talk Fourier" to your colleagues (families and friends), and is an essential part of the tool kit of any scientist who works with data.  

Likewise, understanding the properties (including symmetry) of the Fourier Transform is important for everything that follows.  Of particular importance are functions like the $\delta$, Gaussian, Exponential Decay and Impulse train.  

Finally, I'd recommend downloading a few other WAV files from the interwebs, plotting them up and looking at their spectra.  You can also grab your favorite piece of geophysical data and look at it as well! I guarantee that there will be interesting things in there that you wouldn't have noticed if you were just hanging out in the time domain.

<b>Additional references</b>

1. Bracewell, R.N., 1965, The Fourier Transform and Applications, McGraw-Hill, New York.
2. James, J.F., 2011, A Student's Guide to Fourier Transforms, 3rd ed, Cambridge University Press.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 5.1 &mdash; The marine source ghost</b>

A marine air gun fires at depth $d=6$ m in water with sound speed $v=1500$ m/s. The recorded downgoing pulse is the direct wavelet $w(t)$ plus its reflection from the sea surface, which has reflection coefficient $-1$ and arrives $\Delta t=2d/v$ later: $s(t)=w(t)-w(t-\Delta t)$.

<b>(a)</b> Using linearity and the translation theorem (equations 30 and 31), write $\widehat S(\omega)$ in terms of $\widehat W(\omega)$. Evaluate the "ghost factor" $|1-e^{-i\omega\Delta t}|$ at 10, 30, 62.5 and 125 Hz.

<b>(b)</b> Show that $|1-e^{-i\omega\Delta t}|=2|\sin(\omega\Delta t/2)|$ (Module 3 phasor geometry). List the notch frequencies and the frequency of the first maximum. Repeat the 10 Hz value and the first nonzero notch for a tow depth of 3 m.

<b>(c)</b> "A time shift only changes the phase, so the ghost cannot change the magnitude spectrum." What is wrong with this argument?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 5.1</b></summary>

<b>(a)</b> $\widehat S=\widehat W\,(1-e^{-i\omega\Delta t})$, with $\Delta t=2(6)/1500=8$ ms. Ghost factor: 0.497 (10 Hz), 1.369 (30 Hz), 2.000 (62.5 Hz), 0.000 (125 Hz).

<b>(b)</b> $1-e^{-i\theta}=e^{-i\theta/2}(e^{i\theta/2}-e^{-i\theta/2})=2i\,e^{-i\theta/2}\sin(\theta/2)$, so the magnitude is $2|\sin(\pi f\Delta t)|$. Notches at $f=n/\Delta t$ = 0, 125, 250 Hz, &hellip;; first maximum at $1/(2\Delta t)=62.5$ Hz. At 3 m, $\Delta t=4$ ms: the first nonzero notch moves up to 250 Hz, but the 10 Hz factor falls to 0.251. A shallower tow widens the band upward and loses low frequencies.

<b>(c)</b> Each copy alone has the magnitude of $w$, but the record is the <i>sum</i> of two copies, and the magnitude of a sum depends on the relative phase $\omega\Delta t$ of its terms. With the sign flip from the $-1$ reflection, the copies cancel where $\omega\Delta t$ is a multiple of $2\pi$ and add where it is an odd multiple of $\pi$. This is two-source interference, as for a wave reflected with phase inversion in PHGN 200.
</details>

<div class="alert alert-warning">
<b>Microquestion 5.2 &mdash; Hammer blows and bandwidth</b>

A sledgehammer strikes a steel plate. Model the force as a boxcar of total duration $\tau$ and height $F_0$. On hard ground the contact lasts $\tau=4$ ms; on soft soil it lasts $\tau=20$ ms. In both cases the hammer delivers the same impulse $J=F_0\tau=40$ N&middot;s.

<b>(a)</b> Using Example 1a (half-width $T_0=\tau/2$), find the first zero of the force spectrum, in hertz, for each case. What are $F_0$ and $\widehat F(0)$ in each case?

<b>(b)</b> A smooth pulse $g(t)=e^{-t^2/a^2}$ has the transform of equation (73). Show that the product of its 1/e half-widths in time and angular frequency is 2 for any $a$, and give the 1/e half-width in hertz for $a=2$ ms and $a=10$ ms.

<b>(c)</b> A student wants a 4 ms source pulse whose spectrum is confined below 50 Hz "by using a Gaussian instead of a boxcar." Is that possible? Use equation (33).
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 5.2</b></summary>

<b>(a)</b> $\widehat F(\omega)=2F_0\sin(\omega T_0)/\omega$ has its first zero at $\omega T_0=\pi$, i.e., $f=1/(2T_0)=1/\tau$: 250 Hz on hard ground and 50 Hz on soft soil. $F_0=J/\tau$ = 10 kN and 2 kN. $\widehat F(0)=\int F\,dt=J$ = 40 N&middot;s in both cases (equation 35): this is the impulse of PHGN 100.

<b>(b)</b> $|\widehat G|=a\sqrt\pi\,e^{-a^2\omega^2/4}$ falls to 1/e of its peak at $\omega_e=2/a$, and $g$ falls to 1/e at $t=a$, so the product is $a\cdot 2/a=2$. In hertz, $f_e=1/(\pi a)$: 159 Hz for $a=2$ ms and 31.8 Hz for $a=10$ ms (Figure 5.8 shows the same trade-off).

<b>(c)</b> No. Changing the pulse shape changes the constant ($\pi$ for the boxcar\'s first zero, 2 for the Gaussian\'s 1/e widths), but for any fixed shape the scaling theorem ties duration and bandwidth inversely. A Gaussian of 2 ms half-width (about 4 ms long) already extends to about 159 Hz at its 1/e point, so the only way to concentrate the energy below 50 Hz is a longer pulse, as on soft soil.
</details>

<div class="alert alert-warning">
<b>Microquestion 5.3 &mdash; Where is that peak?</b>

Figure 5.11 uses the first $N=400$ samples of the CAPshot record, sampled at $\Delta t=4$ ms, with <code>freqs = np.fft.fftfreq(400, 0.004)</code>.

<b>(a)</b> What is the frequency spacing $\Delta f$? What frequencies do indices 1, 199, 200 and 399 of <code>freqs</code> hold?

<b>(b)</b> Using equation (83), show that the DTFT of the sampled trace satisfies $\widehat{X_s}(\omega+2\pi/\Delta t)=\widehat{X_s}(\omega)$ (Module 3: $e^{-i2\pi n}=1$). What is this period in hertz, and why does index 200 hold $-125$ Hz rather than $+125$ Hz?

<b>(c)</b> A student plots <code>abs(np.fft.fft(trace))</code> against <code>np.arange(400)*0.625</code> and reports "a strong peak at 225 Hz" at index 360. What frequency is it really, and why does a matching peak appear elsewhere?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 5.3</b></summary>

<b>(a)</b> $\Delta f=1/(N\Delta t)=1/(1.6\ \mathrm{s})=0.625$ Hz. Index 1: 0.625 Hz; 199: 124.375 Hz; 200: $-125$ Hz; 399: $-0.625$ Hz.

<b>(b)</b> Replacing $\omega$ by $\omega+2\pi/\Delta t$ multiplies each term of equation (83) by $e^{-i2\pi n}=1$ (MATH 213: the sum is unchanged term by term). The period is $1/\Delta t=250$ Hz (1570.8 rad/s). The spectrum on $[-125, 125)$ Hz therefore describes every frequency; $+125$ Hz and $-125$ Hz are the same point of a periodic spectrum, and NumPy stores it as $-125$ Hz.

<b>(c)</b> <code>fftfreq</code> gives index 360 &rarr; $-25$ Hz. The record is real, so $|\widehat X(-\omega)|=|\widehat X(\omega)|$ (Table 5.1, row 1), and the same peak appears at $+25$ Hz (index 40). The student\'s axis is wrong above index 199; plot against <code>np.fft.fftshift(freqs)</code>, or use <code>np.fft.rfftfreq</code> for a real signal.
</details>